# Week 1 · Day 4 — Tokenization with tiktoken

LLMs don't read text — they read **tokens**: integer IDs drawn from a fixed
vocabulary. `tiktoken` is OpenAI's tokenizer, and this notebook walks both
directions:

- **encode**: text → list of token IDs (what the model actually consumes)
- **decode**: token IDs → text

The important takeaway: cost, context-window limits, and rate limits are all
counted in *tokens* — not words or characters.

## 1. Encoding: text → token IDs

`encoding_for_model` loads the vocabulary **and** merge rules for that model
(the GPT-4o / GPT-5 family uses one called `o200k_base`, ~200k entries).
`encode` then applies **Byte Pair Encoding (BPE)**: split the text into chunks,
convert each to UTF-8 bytes, repeatedly merge the most frequent byte-pairs, and
map every final chunk to its integer ID.

In [1]:
import tiktoken

encoding = tiktoken.encoding_for_model("gpt-5-nano")
tokens = encoding.encode("Hey, I just watched Narcos season 2 yesterday, it was amazing!")


In [2]:
tokens

[25216,
 11,
 357,
 1327,
 25301,
 32488,
 10732,
 5217,
 220,
 17,
 18809,
 11,
 480,
 673,
 8467,
 0]

## 2. Decoding: token IDs → text

`decode` reverses the lookup: each ID maps back to a byte sequence, which is
concatenated and UTF-8 decoded. Decoding the tokens **one at a time** below
exposes exactly how the sentence was split, and reveals three BPE quirks:

- **Subwords, not words** — "Narcos" isn't common enough to be a single token,
  so it splits into `' Nar'` + `'cos'`. Frequent words like `'Hey'` stay whole.
- **Spaces ride with the next word** — `' I'`, `' just'`, `' watched'` all
  include their leading space.
- **Digits and punctuation stand alone** — the `2`, the commas, and the `!`
  are each their own token.

In [3]:
for token in tokens:
    text = encoding.decode([token])
    print(f"Token: {token}, Text: '{text}'")

Token: 25216, Text: 'Hey'
Token: 11, Text: ','
Token: 357, Text: ' I'
Token: 1327, Text: ' just'
Token: 25301, Text: ' watched'
Token: 32488, Text: ' Nar'
Token: 10732, Text: 'cos'
Token: 5217, Text: ' season'
Token: 220, Text: ' '
Token: 17, Text: '2'
Token: 18809, Text: ' yesterday'
Token: 11, Text: ','
Token: 480, Text: ' it'
Token: 673, Text: ' was'
Token: 8467, Text: ' amazing'
Token: 0, Text: '!'


## 3. Any ID is just a vocabulary entry

`decode` doesn't need a matching `encode` first — the vocabulary is a flat
lookup table, so *any* valid ID resolves to some text fragment. Below, `999`
happens to map to `'her'`.

> **Note:** decoding tokens individually only looks clean because each token
> here is complete, valid UTF-8. A single character (e.g. an emoji) can span
> several tokens — decoding one of those alone yields a broken half-character.
> Only `decode(encode(text)) == text` over the **full** list is guaranteed
> lossless.

In [4]:
encoding.decode([999])

'her'